# Flight Delay and Cancellation Analysis (Part 1)

This notebook contains one consolidated and functional Part 1 workflow.
It merges the previous organized narrative with the currently working implementation.

## Notebook Goals

- Build a reliable preprocessing and feature-engineering pipeline.
- Perform exploratory analysis and dimensionality reduction.
- Run hypothesis testing to statistically validate feature relevance.
- Save intermediate and final checkpoints for reproducibility.

## Execution Phases

1. Problem formulation
2. Data loading and cleansing
3. Feature engineering and transformation
4. EDA and dimensionality reduction
5. Hypothesis testing and artifact export

> Tip: set `NROWS = 10000` for a quick run, or `NROWS = None` for the full dataset.


In [1]:
from pathlib import Path
import sys
import logging

import pandas as pd

# Resolve project root dynamically to avoid hardcoded absolute paths.
project_root = Path.cwd().resolve()
while project_root != project_root.parent and project_root.name != "DataScience_IA":
    project_root = project_root.parent

if project_root.name != "DataScience_IA":
    raise FileNotFoundError("Could not locate project root folder 'DataScience_IA'.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Project_Code.PythonCode.DataPreProcessor.FlightDataCleaner import FlightDataCleaner
from Project_Code.PythonCode.FeatureEngeneering.FlightFeatureEngineer import FlightFeatureEngineer
from Project_Code.PythonCode.EDA.FlightEDA import FlightEDA
from Project_Code.PythonCode.Util.DataVisualization import DataVisualization
from Project_Code.PythonCode.Util.DataLoader import DataLoader
from Project_Code.PythonCode.HypothesisTesting.HypothesisTester import HypothesisTester


dataset_path = project_root / "DataSet" / "flights_sample_3m.csv"
output_dir = project_root / "Output_Files"
output_dir.mkdir(parents=True, exist_ok=True)

# DataLoader now auto-downloads the dataset if this local CSV is missing.
# Keep dataset_path as the target location where the CSV should exist/be created.

# Logger setup used across all notebook phases.
logger = logging.getLogger("part1_notebook")
logger.setLevel(logging.INFO)
if logger.handlers:
    logger.handlers.clear()

file_handler = logging.FileHandler(output_dir / "pipeline_part1_notebook.log", encoding="utf-8")
stream_handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

# Runtime configuration.
NROWS = 10000
TEST_SIZE = 0.2
RANDOM_STATE = 42

print("Project root:", project_root)
print("Dataset path:", dataset_path)
print("Output dir:", output_dir)


Project root: C:\Users\angel\Documents\Github\Data Science\DataScience_IA
Dataset path: C:\Users\angel\Documents\Github\Data Science\DataScience_IA\DataSet\flights_sample_3m.csv
Output dir: C:\Users\angel\Documents\Github\Data Science\DataScience_IA\Output_Files


## Phase 1 - Problem Formulation

This project studies domestic US flight operations (2019-2023) with two predictive targets and one exploratory objective.

### Main Analytical Objectives

1. **Regression task**: predict `ARR_DELAY` (arrival delay in minutes).
2. **Classification task**: predict `DELAY_CLASS` after feature engineering.
3. **Exploratory objective**: identify operational patterns and group-level behavior.

### Leakage-Aware Perspective

A key requirement is to avoid data leakage by excluding post-departure and post-arrival signals during modeling preparation.
This ensures that engineered features represent information realistically available before departure.


In [2]:
phase1_summary = pd.DataFrame(
    {
        "Objective": ["Regression", "Classification", "Pattern Discovery"],
        "Target / Focus": ["ARR_DELAY", "DELAY_CLASS", "EDA + Reduction"],
    }
)
phase1_summary


,Objective,Target / Focus
0,Regression,ARR_DELAY
1,Classification,DELAY_CLASS
2,Pattern Discovery,EDA + Reduction


## Phase 2 - Data Loading and Cleansing

This phase standardizes the raw data preparation pipeline and persists an intermediate checkpoint.

### Steps

1. Load data and create train/test split metadata.
2. Clean raw records (cancelled/diverted filtering, leakage column removal, outlier treatment, null handling).
3. Validate resulting cleaned dataset dimensions and quality.
4. Save checkpoint after feature generation (next phase).


In [3]:
logger.info("\n" + "=" * 80)
logger.info("PHASE 2: DATA LOADING AND CLEANSING")
logger.info("=" * 80)

# STEP 1: Data load and train/test split
logger.info("[STEP 1] Loading dataset and splitting train/test...")
loader = DataLoader(str(dataset_path), test_size=TEST_SIZE, random_state=RANDOM_STATE)
data_train, data_test, target_train, target_test = loader.load_data(nrows=NROWS)

print("Train shape:", data_train.shape)
print("Test shape:", data_test.shape)
print("Target train shape:", target_train.shape if target_train is not None else None)
print("Target test shape:", target_test.shape if target_test is not None else None)

data_train.head(3)


2026-03-20 17:07:26,487 - INFO - 
2026-03-20 17:07:26,488 - INFO - PHASE 2: DATA LOADING AND CLEANSING
2026-03-20 17:07:26,489 - INFO - ================================================================================
2026-03-20 17:07:26,490 - INFO - [STEP 1] Loading dataset and splitting train/test...


Dataset not found at: C:\Users\angel\Documents\Github\Data Science\DataScience_IA\DataSet\flights_sample_3m.csv
Attempting to download dataset from KaggleHub...


ImportError: Missing dependency 'kagglehub'. Install it with: pip install kagglehub[pandas-datasets]

In [ ]:
# STEP 2: Raw cleaning
logger.info("[STEP 2] Cleaning raw data...")
cleaner = FlightDataCleaner(file_path=str(dataset_path))
df_clean = cleaner.load_and_clean(nrows=NROWS)

print("Clean shape:", df_clean.shape)
print("Numeric missing values after cleaning:", int(df_clean.select_dtypes(include=["number"]).isnull().sum().sum()))
df_clean.head(3)


## Phase 3 - Feature Engineering and Transformation

This phase converts cleaned operational fields into model-ready features.

### What is created in this step

- Temporal behavior features
- Route and interaction features
- Delay classification target (`DELAY_CLASS`)
- Encoded categorical columns
- Normalized numeric predictors


In [ ]:
logger.info("\n" + "=" * 80)
logger.info("PHASE 3: FEATURE ENGINEERING")
logger.info("=" * 80)

# STEP 3: Generate features
logger.info("[STEP 3] Generating engineered features...")
engineer = FlightFeatureEngineer(df_clean)
df_features = engineer.generate_features()

# STEP 4: Encode categorical features
logger.info("[STEP 4] Encoding categorical variables...")
df_features = engineer.encode_categorical()

# STEP 5: Normalize numeric features
logger.info("[STEP 5] Normalizing numeric features...")
df_features = engineer.normalize_features()

print("Features shape:", df_features.shape)
print("Total columns:", len(df_features.columns))
print("DELAY_CLASS present:", "DELAY_CLASS" in df_features.columns)
df_features.head(3)


In [ ]:
# STEP 6: Intermediate checkpoint
logger.info("[STEP 6] Saving cleaned+features checkpoint...")
loader.data = df_features
checkpoint_clean_path = dataset_path / "checkpoint_cleaned_features.pkl"
loader.save_checkpoint(str(checkpoint_clean_path))

print("Checkpoint created:", checkpoint_clean_path.exists())
print("Checkpoint path:", checkpoint_clean_path)


## Phase 4 - Exploratory Data Analysis and Dimensionality Reduction

This section combines analytical diagnostics with visual diagnostics.

### Analytical diagnostics

- Distribution and descriptive summaries
- Variable ranges and correlation structure
- Data quality checks

### Dimensionality reduction

- PCA for linear structure
- UMAP (or t-SNE fallback) for non-linear structure


In [ ]:
logger.info("\n" + "=" * 80)
logger.info("PHASE 4: EDA AND DIMENSIONALITY REDUCTION")
logger.info("=" * 80)

# STEP 7: Run full EDA routine
logger.info("[STEP 7] Running full EDA...")
eda = FlightEDA(df_features, target_col="ARR_DELAY", output_dir=output_dir, group_col="DELAY_CLASS")
eda_report = eda.perform_eda()

print("Missing values total:", int(eda_report["quality"]["missing_values"].sum()))
print("Duplicate rows:", int(eda_report["quality"]["duplicate_count"]))
print("Top outlier counts:")
print(eda_report["quality"]["outlier_count"].head(5))


In [ ]:
# STEP 8: PCA
logger.info("[STEP 8] Running PCA...")
pca_components = eda.run_pca(n_components=2, explained_variance_threshold=0.8)

print("PCA type:", type(pca_components))
print("PCA shape:", pca_components.shape)


### Detailed Analytical EDA Outputs

The following cells expose explicit analytical components from the EDA class.
These are useful when building a written report because each diagnostic can be cited independently.


In [ ]:
eda.describe_variables()


In [ ]:
eda.determine_range()


In [ ]:
eda.assess_correlation()


In [ ]:
eda.assess_quality()


### Visual EDA Outputs

These cells generate key plots used for interpretation and documentation.
All figures are written to `Output_Files`.


In [ ]:
eda.viz.plot_distributions()


In [ ]:
eda.viz.plot_correlation_matrix(filename="eda_correlation_matrix.png")


In [ ]:
eda.viz.plot_boxplots(columns=eda.numeric_cols, filename="eda_boxplots.png")


In [ ]:
eda.viz.plot_target_distribution(filename="eda_target_distribution.png")


In [ ]:
# STEP 9: UMAP/t-SNE
logger.info("[STEP 9] Running UMAP/t-SNE...")
umap_components = eda.run_umap_or_tsne(n_components=2, use_umap=True)

print("UMAP/t-SNE type:", type(umap_components))
print("UMAP/t-SNE shape:", umap_components.shape)


In [ ]:
# STEP 10: Additional visual diagnostics
logger.info("[STEP 10] Generating additional visual diagnostics...")
viz = DataVisualization(df_features, output_dir=output_dir)
viz.plot_heatmap_top_correlations(top_n=20)

if "DELAY_CLASS" in df_features.columns:
    focus_cols = ["DISTANCE", "CRS_ELAPSED_TIME", "PLANNED_SPEED_MPM", "ARR_DELAY"]
    viz.plot_grouped_feature_distributions(
        columns=focus_cols,
        group_col="DELAY_CLASS",
        filename="viz_grouped_distributions_delay_class.png",
    )
    viz.plot_grouped_boxplots(
        columns=focus_cols,
        group_col="DELAY_CLASS",
        filename="viz_grouped_boxplots_delay_class.png",
    )

generated_eda_files = sorted([p.name for p in output_dir.glob("eda_*.png")])
print("Generated EDA files:")
for name in generated_eda_files:
    print("-", name)


## Phase 5 - Hypothesis Testing and Final Checkpoint

This final phase provides statistical evidence for feature differences across delay classes.

### Statistical output goals

1. Run a full battery of tests per numeric feature.
2. Export each test family as CSV for reporting and traceability.
3. Save the final checkpoint artifact.


In [ ]:
logger.info("\n" + "=" * 80)
logger.info("PHASE 5: HYPOTHESIS TESTING AND FINAL CHECKPOINT")
logger.info("=" * 80)

# STEP 11: Hypothesis testing
logger.info("[STEP 11] Running statistical tests...")
hypothesis_tester = HypothesisTester(
    data=df_features,
    labels=df_features["DELAY_CLASS"],
    target_col="DELAY_CLASS",
    verbose=False,
)

summary_report = hypothesis_tester.generate_summary_report()


In [ ]:
# STEP 12: Export statistical reports
logger.info("[STEP 12] Exporting statistical reports...")
exported_reports = []

for test_name, results_df in summary_report.items():
    if results_df is not None:
        out_csv = output_dir / f"hypothesis_testing_{test_name}.csv"
        results_df.to_csv(out_csv, index=False)
        exported_reports.append(out_csv.name)
        logger.info(f"Report saved: {out_csv.name}")

print("Exported reports:")
for name in exported_reports:
    print("-", name)


In [ ]:
# STEP 13: Final checkpoint
logger.info("[STEP 13] Saving final checkpoint...")
loader.data = df_features
checkpoint_final_path = dataset_path / "checkpoint_part1_complete.pkl"
loader.save_checkpoint(str(checkpoint_final_path))

logger.info("=" * 80)
logger.info("PIPELINE PART 1 COMPLETED SUCCESSFULLY (NOTEBOOK)")
logger.info("=" * 80)

print("Final checkpoint created:", checkpoint_final_path.exists())
print("Final checkpoint path:", checkpoint_final_path)
print("Artifacts directory:", output_dir)

